# run_servo_setup: assign servo IDs and calibrate

Run notebook (needs the arm + Waveshare Bus Servo Adapter). Milestone: *Arm assembled, servo
identifiers assigned, calibrated*. Two routes: LeRobot's own tools (recommended first, they
write LeRobot calibration files used by run_teleop) or our `ServoBus` (servo_bus.ipynb), whose
calibration is what the Python/C++ paths use. Keep both consistent.

**TODO**
- [ ] find the adapter port (`ls /dev/ttyACM* /dev/ttyUSB*`), check 1 Mbaud
- [ ] assign IDs one servo at a time (only ONE servo on the bus while writing its ID)
- [ ] calibrate: zero offsets so URDF zero == tick 2048, sign per joint, range of motion
- [ ] save to `gambit_bringup/config/arm_calibration.yaml`; load it in servo_bus.CALIBRATION
- [ ] repeat for the leader arm (LeRobot teleop)

In [ ]:
%run ./servo_bus.ipynb
PORT = "/dev/ttyACM0"   # TODO: set

## Option A: LeRobot (recommended first)

```bash
pip install 'lerobot[feetech]'
lerobot-find-port
lerobot-setup-motors --robot.type=so101_follower --robot.port=/dev/ttyACM0
lerobot-calibrate    --robot.type=so101_follower --robot.port=/dev/ttyACM0 --robot.id=gambit_follower
lerobot-setup-motors --teleop.type=so101_leader --teleop.port=/dev/ttyACM1
lerobot-calibrate    --teleop.type=so101_leader --teleop.port=/dev/ttyACM1 --teleop.id=gambit_leader
```
(Command names change between LeRobot releases; check `lerobot --help` for your version.)

## Option B: our ServoBus

Assign one ID: connect only the new servo (factory ID 1), unlock EEPROM, write `Reg.ID`, lock.

In [ ]:
def assign_id(bus, new_id, current_id=1):
    """TODO: bus.write_u8(current_id, Reg.LOCK, 0); bus.write_u8(current_id, Reg.ID, new_id);
    bus.write_u8(new_id, Reg.LOCK, 1); then bus.ping(new_id). Verify addresses first."""
    raise NotImplementedError

def scan(bus, ids=range(0, 20)):
    return [i for i in ids if bus.ping(i)]

def record_range(bus, seconds=20):
    """TODO: torque off, ask the user to move every joint through its full range, read
    positions at ~50 Hz, return {joint: (min_ticks, max_ticks)}; with the arm held at the URDF
    zero pose compute offset_ticks = 2048 - raw for each joint (and decide signs)."""
    raise NotImplementedError

# with ServoBus(PORT) as bus:
#     print(scan(bus))
#     assign_id(bus, 2)

In [ ]:
def save_calibration(cal, path=CONFIG_DIR / "arm_calibration.yaml"):
    import yaml
    data = {n: {"offset_ticks": c.offset_ticks, "sign": c.sign} for n, c in cal.items()}
    with open(path, "w") as f:
        yaml.safe_dump(data, f)
    return path

# TODO: after calibration: save_calibration(CALIBRATION)